# RWB service v2

Current local service: point forecasts, calibrated quantiles, reconciled half-hour flows and shared snapshots. Historical periods are reused; this is not a new holdout. Prepare the bundle with `python scripts/prepare_service_v2.py` and the report with `python scripts/validate_service_v2.py` before running this notebook.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from rwb.data import load_panel
from rwb.features import cumsums
from rwb.service import Service
from rwb.snapshot import SnapshotStore, route_status, network_summary, evidence
panel = load_panel()
sums = cumsums(panel)
service = Service.load(ROOT / "models/prod/service_v2")
store = SnapshotStore(ROOT / ".local/colhound/snapshots", ROOT / "reports/colhound/service_v2/report.json")
snapshot, cache_hit = store.get_or_create(service, panel, sums, panel.n_times - 1)
print("snapshot", snapshot["snapshot_id"], "cache_hit", cache_hit)


In [ ]:
route = route_status(snapshot, snapshot["routes"][0]["route_id"])
display(pd.DataFrame({"timestamp": snapshot["forecast_times"], **{key: route["forecast"][key] for key in ["point_forecast", "p10", "p50", "p90", "inflow_point"]}}))
print("Dock state:", snapshot["dock_state_source"])
print("Calibration:", snapshot["calibration_metadata"])


In [ ]:
display(pd.read_csv(ROOT / "reports/colhound/service_v2/summary.csv"))
print(evidence(snapshot, "validation"))
print(network_summary(snapshot))
